# Libraries + OOP para Trading

Del `import` a un mini sistema reusable.

## Objetivo de la sesion

- entender que una libreria es codigo que ya existe y que `import` te da acceso a ese trabajo
- usar `pandas` de forma ligera para leer una tabla y calcular algo rapido
- pasar de diccionarios a objetos con `Order`, `Trade` y `PositionTracker`

Antes de ejecutar, intenta predecir que deberia salir. Si algo te sorprende, ahi esta la leccion.


## 1. Una libreria te amplia el espacio de trabajo

Una libreria es codigo que ya escribio otra persona. `import pandas as pd` significa: quiero usar esa caja de herramientas desde mi programa.

**Antes de ejecutar:** cuantas filas tendra `trades_df`? Que columnas esperas ver?


In [1]:
import pandas as pd

trades_df = pd.DataFrame(
    [
        {"symbol": "BTCUSDT", "side": "buy", "price": 99980, "size": 0.10},
        {"symbol": "BTCUSDT", "side": "sell", "price": 100020, "size": 0.08},
        {"symbol": "ETHUSDT", "side": "buy", "price": 3520, "size": 1.40},
    ]
)

print("rows:", len(trades_df))
print("columns:", list(trades_df.columns))
trades_df

rows: 3
columns: ['symbol', 'side', 'price', 'size']


,symbol,side,price,size
0,BTCUSDT,buy,99980,0.10
1,BTCUSDT,sell,100020,0.08
2,ETHUSDT,buy,3520,1.40


## 2. `pandas` gana velocidad en tablas

Aqui no queremos volvernos expertos en data analysis. Solo sentir una ganancia inmediata: crear una columna y resumir algo con muy poco codigo.

**Antes de ejecutar:** que fila tendra mayor `notional`?


In [2]:
trades_df["notional"] = trades_df["price"] * trades_df["size"]
volume_by_side = trades_df.groupby("side")["size"].sum()

print(trades_df[["symbol", "side", "price", "size", "notional"]])
print()
print("volume_by_side:")
print(volume_by_side)

    symbol  side   price  size  notional
0  BTCUSDT   buy   99980  0.10    9998.0
1  BTCUSDT  sell  100020  0.08    8001.6
2  ETHUSDT   buy    3520  1.40    4928.0

volume_by_side:
side
buy     1.50
sell    0.08
Name: size, dtype: float64


## 3. Pero una tabla no organiza todo tu sistema

Una libreria te ayuda a trabajar con datos. Otra cosa distinta es como organizas tu propio codigo. Si una orden empieza a necesitar comportamiento, repetir formulas por fuera se vuelve incomodo.


In [3]:
order_dict = {
    "symbol": "BTCUSDT",
    "side": "buy",
    "price": 99980,
    "size": 0.10,
}

notional = order_dict["price"] * order_dict["size"]
description = f"{order_dict['side']} {order_dict['size']:.2f} {order_dict['symbol']} @ {order_dict['price']}"

print(order_dict)
print("notional:", notional)
print("description:", description)

{'symbol': 'BTCUSDT', 'side': 'buy', 'price': 99980, 'size': 0.1}
notional: 9998.0
description: buy 0.10 BTCUSDT @ 99980


## 4. Primera clase propia: `Order`

Una clase junta datos y comportamiento. La orden ya no solo guarda `price` y `size`; tambien sabe calcular su propio nocional y describirse.

**Antes de ejecutar:** que devolvera `order.notional()`?


In [4]:
class Order:
    def __init__(self, symbol, side, price, size):
        self.symbol = symbol
        self.side = side
        self.price = price
        self.size = size

    def notional(self):
        return self.price * self.size

    def describe(self):
        return f"{self.side} {self.size:.2f} {self.symbol} @ {self.price}"


order = Order("BTCUSDT", "buy", 99980, 0.10)

print("symbol:", order.symbol)
print("side:", order.side)
print("notional:", order.notional())
print("describe:", order.describe())

symbol: BTCUSDT
side: buy
notional: 9998.0
describe: buy 0.10 BTCUSDT @ 99980


## 5. Otro objeto: `Trade`

Una orden expresa intencion. Un trade expresa ejecucion. Es util separarlos para que el sistema empiece a parecerse a piezas reales.


In [5]:
class Trade:
    def __init__(self, symbol, side, price, size):
        self.symbol = symbol
        self.side = side
        self.price = price
        self.size = size

    def cash_flow(self):
        signed = -1 if self.side == "buy" else 1
        return signed * self.price * self.size


fill = Trade("BTCUSDT", "buy", 100000, 0.10)

print("fill side:", fill.side)
print("fill size:", fill.size)
print("cash_flow:", fill.cash_flow())

fill side: buy
fill size: 0.1
cash_flow: -10000.0


## 6. Estado reusable: `PositionTracker`

Cuando aparece estado como `cash` y `position`, agrupar la logica en un objeto ayuda aun mas.

**Pregunta:** si aplico una compra, que deberia pasar con `cash` y con `position`?


In [6]:
class PositionTracker:
    def __init__(self):
        self.cash = 0.0
        self.position = 0.0

    def apply_trade(self, trade):
        if trade.side == "buy":
            self.cash = self.cash - trade.price * trade.size
            self.position = self.position + trade.size
        else:
            self.cash = self.cash + trade.price * trade.size
            self.position = self.position - trade.size

    def equity(self, mark_price):
        return self.cash + self.position * mark_price


tracker = PositionTracker()
tracker.apply_trade(Trade("BTCUSDT", "buy", 100000, 0.10))
tracker.apply_trade(Trade("BTCUSDT", "sell", 100120, 0.04))

print("cash:", tracker.cash)
print("position:", tracker.position)
print("equity @ 100050:", tracker.equity(100050))

cash: -5995.2
position: 0.060000000000000005
equity @ 100050: 7.800000000001091


## 7. Mini historia completa: de ordenes a equity

Ya podemos combinar libreria, clases, objetos y estado en una pequena historia de trading.


In [7]:
orders = [
    Order("BTCUSDT", "buy", 99980, 0.10),
    Order("BTCUSDT", "sell", 100040, 0.06),
    Order("ETHUSDT", "buy", 3520, 1.25),
]

fills = [
    Trade("BTCUSDT", "buy", 100000, 0.10),
    Trade("BTCUSDT", "sell", 100110, 0.03),
    Trade("BTCUSDT", "sell", 100140, 0.02),
]

tracker = PositionTracker()
for trade in fills:
    tracker.apply_trade(trade)

print("orders:")
for order_item in orders:
    print("-", order_item.describe(), "| notional:", order_item.notional())

print()
print("cash:", tracker.cash)
print("position:", tracker.position)
print("equity @ 100090:", tracker.equity(100090))

orders:
- buy 0.10 BTCUSDT @ 99980 | notional: 9998.0
- sell 0.06 BTCUSDT @ 100040 | notional: 6002.4
- buy 1.25 ETHUSDT @ 3520 | notional: 4400.0

cash: -4993.900000000001
position: 0.05
equity @ 100090: 10.599999999999454


## 8. Tu turno

Crea:

- `eth_order` como orden de compra de ETH
- `eth_trade` como trade de compra de ETH
- un nuevo `tracker_two` que aplique ese trade
- `eth_equity` usando un `mark_price` de `3550`

**Pista:** usa exactamente las clases que ya hemos definido.


In [8]:
eth_order = None
eth_trade = None
tracker_two = None
eth_equity = None

print("TODO -> crea `eth_order`, `eth_trade`, `tracker_two` y `eth_equity`")

TODO -> crea `eth_order`, `eth_trade`, `tracker_two` y `eth_equity`


## 9. Una posible solucion

Comparala con tu enfoque. Lo importante no es copiarla, sino ver si entendiste como se conectan objeto y estado.


In [9]:
eth_order = Order("ETHUSDT", "buy", 3525, 1.50)
eth_trade = Trade("ETHUSDT", "buy", 3525, 1.50)
tracker_two = PositionTracker()
tracker_two.apply_trade(eth_trade)
eth_equity = tracker_two.equity(3550)

print("eth order:", eth_order.describe())
print("eth order notional:", eth_order.notional())
print("cash:", tracker_two.cash)
print("position:", tracker_two.position)
print("eth_equity:", eth_equity)

eth order: buy 1.50 ETHUSDT @ 3525
eth order notional: 5287.5
cash: -5287.5
position: 1.5
eth_equity: 37.5


## Cierre

Que deberia haberte quedado claro:

- **Librerias:** `import` te da acceso a herramientas que ya existen, y `pandas` es una forma rapida de trabajar con tablas.
- **Objetos:** una clase junta datos y comportamiento con una forma reusable.
- **Sistema:** con `Order`, `Trade` y `PositionTracker` el codigo empieza a parecerse a piezas de mercado de verdad.

La siguiente capa natural es usar estas ideas sobre datos y estructuras de mercado cada vez mas realistas.
